In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, IntSlider, FloatSlider, HBox, Layout, VBox, HTML, GridBox
from IPython.display import display

# ============================================================
# WHITE-NOISE PARAMETERS
# ============================================================

np.random.seed(84)

M_max = 200
N_max = 1000

# Fixed unit-variance Gaussian ensemble
W = np.random.randn(M_max, N_max)

# ============================================================
# INTERACTIVE FUNCTION
# ============================================================

def plot_white_noise(N=500, M=80, sigma=1.0):

    # --------------------------------------------------------
    # SELECT FINITE ENSEMBLE
    # --------------------------------------------------------

    X = sigma * W[:M, :N]

    # One realization for the time-domain and autocorrelation graphs
    realization = X[0, :]

    # --------------------------------------------------------
    # REMOVE FINITE-SAMPLE MEAN
    # --------------------------------------------------------

    realization_centered = realization - np.mean(realization)

    X_centered = X - np.mean(X, axis=1, keepdims=True)

    # --------------------------------------------------------
    # AUTOCORRELATION ESTIMATE
    # FROM ONE REALIZATION
    # --------------------------------------------------------

    max_lag = 50

    R_positive = np.zeros(max_lag + 1)

    for k in range(max_lag + 1):

        R_positive[k] = np.mean(
            realization_centered[:N - k] *
            realization_centered[k:]
        )

    R_negative = R_positive[1:][::-1]

    R_estimated = np.concatenate(
        [
            R_negative,
            R_positive
        ]
    )

    lags = np.arange(
        -max_lag,
        max_lag + 1
    )

    # --------------------------------------------------------
    # THEORETICAL AUTOCORRELATION
    # --------------------------------------------------------

    R_theoretical = np.zeros_like(
        lags,
        dtype=float
    )

    R_theoretical[max_lag] = sigma ** 2

    # --------------------------------------------------------
    # PSD ESTIMATE
    # ENSEMBLE-AVERAGED PERIODOGRAM
    # --------------------------------------------------------

    n_fft = 2048

    periodograms = np.zeros((M, n_fft))

    for m in range(M):

        X_fft = np.fft.fftshift(
            np.fft.fft(
                X_centered[m, :],
                n=n_fft
            )
        )

        periodograms[m, :] = (
            np.abs(X_fft) ** 2
        ) / N

    PSD_estimated = np.mean(
        periodograms,
        axis=0
    )

    omega = np.linspace(
        -np.pi,
        np.pi,
        n_fft,
        endpoint=False
    )

    PSD_theoretical = np.full(
        n_fft,
        sigma ** 2
    )

    # --------------------------------------------------------
    # FIGURE
    # --------------------------------------------------------

    fig, (ax1, ax2, ax3) = plt.subplots(
        3,
        1,
        figsize=(8.0, 7.0)
    )

    # ========================================================
    # GRAPH 1:
    # WHITE-NOISE REALIZATION
    # ========================================================

    ax1.plot(
        np.arange(N),
        realization,
        linewidth=1.0
    )

    ax1.set_xlim(
        0,
        N - 1
    )

    ax1.set_ylim(
        -8.0,
        8.0
    )

    ax1.set_xlabel(
        'Time index n',
        fontsize=11
    )

    ax1.set_ylabel(
        'x[n]',
        fontsize=11
    )

    ax1.set_title(
        f'White-Noise Realization, σ = {sigma:.2f}',
        fontsize=12,
        pad=7
    )

    ax1.tick_params(
        axis='both',
        labelsize=9
    )

    ax1.grid(
        True,
        linestyle=':',
        alpha=0.6
    )

    # ========================================================
    # GRAPH 2:
    # AUTOCORRELATION
    # ========================================================

    ax2.plot(
        lags,
        R_estimated,
        linewidth=1.8,
        label='Estimated autocorrelation'
    )

    ax2.plot(
        lags,
        R_theoretical,
        linestyle='--',
        linewidth=1.5,
        label='Theoretical autocorrelation'
    )

    ax2.axhline(
        0,
        color='k',
        linewidth=0.8
    )

    ax2.axvline(
        0,
        color='k',
        linestyle=':',
        linewidth=0.8
    )

    ax2.set_xlim(
        -max_lag,
        max_lag
    )

    ax2.set_ylim(
        -0.6,
        5.0
    )

    ax2.set_xlabel(
        'Lag k',
        fontsize=11
    )

    ax2.set_ylabel(
        'Rₓ[k]',
        fontsize=11
    )

    ax2.set_title(
        'White Noise Autocorrelation',
        fontsize=12,
        pad=7
    )

    ax2.tick_params(
        axis='both',
        labelsize=9
    )

    ax2.grid(
        True,
        linestyle=':',
        alpha=0.6
    )

    ax2.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.25),
        ncol=2,
        fontsize=8,
        borderaxespad=0.0
    )

    # ========================================================
    # GRAPH 3:
    # POWER SPECTRAL DENSITY
    # ========================================================

    ax3.plot(
        omega,
        PSD_estimated,
        linewidth=1.8,
        label='Estimated PSD'
    )

    ax3.plot(
        omega,
        PSD_theoretical,
        linestyle='--',
        linewidth=1.5,
        label='Theoretical PSD'
    )

    ax3.set_xlim(
        -np.pi,
        np.pi
    )

    ax3.set_ylim(
        0,
        5.0
    )

    ax3.set_xticks(
        [
            -np.pi,
            -np.pi / 2,
            0,
            np.pi / 2,
            np.pi
        ]
    )

    ax3.set_xticklabels(
        [
            '-π',
            '-π/2',
            '0',
            'π/2',
            'π'
        ]
    )

    ax3.set_xlabel(
        'Angular frequency ω',
        fontsize=11
    )

    ax3.set_ylabel(
        'PSD',
        fontsize=11
    )

    ax3.set_title(
        'White Noise Power Spectral Density',
        fontsize=12,
        pad=7
    )

    ax3.tick_params(
        axis='both',
        labelsize=9
    )

    ax3.grid(
        True,
        linestyle=':',
        alpha=0.6
    )

    # --------------------------------------------------------
    # LEGEND TO THE RIGHT, VERTICAL
    # --------------------------------------------------------

    ax3.legend(
        loc='center left',
        bbox_to_anchor=(1.02, 0.5),
        ncol=1,
        fontsize=8,
        borderaxespad=0.0
    )

    # ========================================================
    # FIGURE SPACING
    # ========================================================

    plt.subplots_adjust(
        left=0.12,
        right=0.78,
        top=0.96,
        bottom=0.10,
        hspace=0.88
    )

    plt.show()

# ============================================================
# SLIDERS
# ============================================================

slider_style = {
    'description_width': '0px'
}

slider_layout = Layout(
    width='100px'
)

N_slider = IntSlider(
    min=100,
    max=1000,
    step=50,
    value=500,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

M_slider = IntSlider(
    min=20,
    max=200,
    step=20,
    value=80,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

sigma_slider = FloatSlider(
    min=0.25,
    max=2.0,
    step=0.05,
    value=1.0,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

# ============================================================
# MAXIMUM VALUE LABELS
# ============================================================

N_max_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">1000</div>'
)

M_max_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">200</div>'
)

sigma_max_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">2.0</div>'
)

# ============================================================
# INTERACTIVE OBJECT
# ============================================================

widget_plot = interactive(
    plot_white_noise,
    N=N_slider,
    M=M_slider,
    sigma=sigma_slider
)

# ============================================================
# DOCUMENTATION
# ============================================================

theory_html = HTML("""
<div style="
    font-family: Arial, sans-serif;
    font-size: 16px;
    line-height: 1.30;
    width: 1050px;
">

<div style="
    font-size: 21px;
    font-weight: bold;
    margin-bottom: 6px;
">
White Noise in the Time and Frequency Domains
</div>

<div style="margin-bottom:5px;">
<b>White noise:</b> a zero-mean random process whose samples are uncorrelated for every nonzero time lag.
</div>

<div style="margin-bottom:5px;">
<b>Autocorrelation:</b> theoretically Rₓ[k] = σ²δ[k], so all correlation is concentrated at k = 0.
</div>

<div style="margin-bottom:5px;">
<b>Power spectral density:</b> theoretically Sₓ(e<sup>jω</sup>) = σ², so white noise has equal average power at all frequencies.
</div>

<div style="margin-bottom:5px;">
<b>Observation length N:</b> increasing N makes the single-realization autocorrelation estimate approach the theoretical impulse more closely.
</div>

<div style="margin-bottom:5px;">
<b>Number of realizations M:</b> increasing M reduces the statistical fluctuations of the ensemble-averaged PSD estimate.
</div>

<div>
<b>This notebook:</b> demonstrates the equivalence between impulse-like autocorrelation and flat power spectral density for a white-noise process.
</div>

</div>
""")

# ============================================================
# EXTRA SPACE BELOW THEORY
# ============================================================

theory_block = VBox(
    [
        theory_html
    ],
    layout=Layout(
        margin='0px 0px 18px 0px'
    )
)

# ============================================================
# LEFT-ALIGNED LABELS
# ============================================================

N_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">Samples N:</div>'
)

M_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">Realizations M:</div>'
)

sigma_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">Std. deviation σ:</div>'
)

# ============================================================
# FIXED THREE-COLUMN GRID
# LABEL | SLIDER | MAXIMUM VALUE
# ============================================================

slider_grid = GridBox(
    children=[
        N_label, N_slider, N_max_label,
        M_label, M_slider, M_max_label,
        sigma_label, sigma_slider, sigma_max_label
    ],
    layout=Layout(
        width='265px',
        grid_template_columns='110px 100px 40px',
        grid_template_rows='30px 30px 30px',
        grid_gap='2px 6px',
        align_items='center',
        overflow='hidden'
    )
)

# ============================================================
# CONTROLS
# ============================================================

controls = VBox(
    [
        slider_grid
    ],
    layout=Layout(
        width='275px',
        min_width='275px',
        align_items='flex-start',
        justify_content='center',
        margin='0px 0px 0px -65px',
        overflow='hidden'
    )
)

# ============================================================
# FIGURE LEFT - CONTROLS RIGHT
# ============================================================

graph_and_controls = HBox(
    [
        widget_plot.children[-1],
        controls
    ],
    layout=Layout(
        width='900px',
        align_items='center',
        justify_content='flex-start',
        overflow='hidden'
    )
)

# ============================================================
# INTERPRETATION BELOW THE FIGURES
# ============================================================

interpretation_html = HTML("""
<div style="
    font-family: Arial, sans-serif;
    font-size: 15px;
    line-height: 1.35;
    width: 1050px;
    margin-top: 14px;
">

<div style="
    font-size: 18px;
    font-weight: bold;
    margin-bottom: 6px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:5px;">
<b>First graph:</b> shows one realization of the white-noise process in the time domain.
</div>

<div style="margin-bottom:5px;">
<b>Second graph:</b> shows the essential time-domain property. The autocorrelation approaches an impulse,
Rₓ[k] = σ²δ[k], so the correlation is approximately zero for every nonzero lag.
</div>

<div style="margin-bottom:5px;">
<b>Third graph:</b> shows the corresponding frequency-domain property. The power spectral density approaches
the constant value Sₓ(e<sup>jω</sup>) = σ².
</div>

<div style="
    margin-top:8px;
    margin-bottom:5px;
    font-size:17px;
    font-weight:bold;
">
Impulse autocorrelation &nbsp;&nbsp; ⇔ &nbsp;&nbsp; Flat power spectral density
</div>

<div>
The first graph is mainly illustrative. The essential Wiener–Khinchin correspondence is demonstrated by the relationship between the second and third graphs.
</div>

</div>
""")

# ============================================================
# COMPLETE LAYOUT
# ============================================================

main_layout = VBox(
    [
        theory_block,
        graph_and_controls,
        interpretation_html
    ],
    layout=Layout(
        width='1100px',
        overflow='hidden'
    )
)

display(main_layout)